In [12]:
import json
from pathlib import Path

import torch
import timm
from PIL import Image
from timm.data import resolve_data_config, create_transform
from torch.utils.data import Dataset, DataLoader

In [13]:
ROOT = Path("/lambda/nfs/neel/Research")

# choose one:
SUBSET = "coco_caption_5k"          # under subsets/
# SUBSET = "vqa_v2_balanced_5k"

MODEL_NAME = "vit_base_patch14_reg4_dinov2"  # small/large/giant also ok
BATCH_SIZE = 16
NUM_WORKERS = 4

META_PATH = ROOT / "subsets" / SUBSET / "metadata.jsonl"
OUT_DIR = ROOT / "runs" / "dinov2" / SUBSET / "regs" / MODEL_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUT_DIR / "regs_index.jsonl"

device = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
# ---- model + correct preprocessing (518x518 + normalize) ----
model = timm.create_model(MODEL_NAME, pretrained=True).eval().to(device)
cfg = resolve_data_config(model.pretrained_cfg, model=model)
transform = create_transform(**cfg)

assert hasattr(model, "reg_token"), "This model has no register tokens."
R = model.reg_token.shape[1]  # should be 4

In [20]:

@torch.no_grad()
def regs_last_layer(model, images_bchw):
    """
    images_bchw: (B,3,H,W) already resized/normalized via timm cfg transform
    returns: (B, R=4, D) register tokens after final block (pre-norm)
    """
    # patch tokens first
    x = model.patch_embed(images_bchw)  # (B,N,D)

    # let timm handle cls/reg + pos embed correctly (and pos interpolation if needed)
    if hasattr(model, "_pos_embed"):
        x = model._pos_embed(x)         # (B, 1+R+N, D) with correct pos logic
    else:
        # fallback: manual, but patch-only pos_embed is common here
        B = x.size(0)
        R = model.reg_token.shape[1]
        cls = model.cls_token.expand(B, -1, -1)
        reg = model.reg_token.expand(B, -1, -1)
        x = torch.cat([cls, reg, x], dim=1)  # (B,1+R+N,D)

        pos = model.pos_embed
        if pos.shape[1] == x.shape[1]:
            x = x + pos
        elif pos.shape[1] == (x.shape[1] - (1 + R)):  # patch-only pos
            x[:, 1+R:, :] = x[:, 1+R:, :] + pos
        elif pos.shape[1] == (x.shape[1] - R):        # cls+patch pos (no regs)
            x[:, 0:1, :] = x[:, 0:1, :] + pos[:, 0:1, :]
            x[:, 1+R:, :] = x[:, 1+R:, :] + pos[:, 1:, :]
        else:
            raise RuntimeError(f"Unhandled pos_embed shape {tuple(pos.shape)} vs tokens {tuple(x.shape)}")

        if hasattr(model, "pos_drop"):
            x = model.pos_drop(x)

    if hasattr(model, "norm_pre") and model.norm_pre is not None:
        x = model.norm_pre(x)

    R = model.reg_token.shape[1]
    for blk in model.blocks:
        x = blk(x)

    return x[:, 1:1+R, :].detach()  # (B,R,D)


In [21]:
# ---- dataset ----
class MetaImages(Dataset):
    def __init__(self, meta_path: Path):
        self.rows = []
        with meta_path.open("r") as f:
            for line in f:
                if line.strip():
                    self.rows.append(json.loads(line))

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        image_id = r.get("image_id", r.get("id", idx))
        img_path = r.get("image_file") or r.get("image_path") or r.get("path")
        if img_path is None:
            raise KeyError("metadata row missing image_file/image_path/path")

        p = Path(img_path)
        if not p.is_absolute():
            p = ROOT / p

        pil = Image.open(p).convert("RGB")
        x = transform(pil)  # (3,518,518) normalized
        return str(image_id), x

In [22]:
def collate(batch):
    ids, xs = zip(*batch)
    return list(ids), torch.stack(xs, dim=0)

In [23]:
ds = MetaImages(META_PATH)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                pin_memory=True, collate_fn=collate)

In [24]:
# ---- run ----
with INDEX_PATH.open("w") as idx_f:
    for ids, x in dl:
        x = x.to(device, non_blocking=True)
        regs = regs_last_layer(model, x).to("cpu", dtype=torch.float16)  # (B,4,D)

        for i, image_id in enumerate(ids):
            out_path = OUT_DIR / f"{image_id}.pt"
            torch.save(regs[i], out_path)  # (4,D) fp16

            idx_f.write(json.dumps({
                "image_id": image_id,
                "reg_path": str(out_path),
                "model": MODEL_NAME,
                "regs": int(R),
                "dtype": "float16",
            }) + "\n")

print(f"Done. Wrote per-image regs to: {OUT_DIR}")
print(f"Index: {INDEX_PATH}")

Done. Wrote per-image regs to: /lambda/nfs/neel/Research/runs/dinov2/coco_caption_5k/regs/vit_base_patch14_reg4_dinov2
Index: /lambda/nfs/neel/Research/runs/dinov2/coco_caption_5k/regs/vit_base_patch14_reg4_dinov2/regs_index.jsonl
